# Recess Ready: Philadelphia

**Question:** each evening, can we predict whether tomorrow will be unsafe for outdoor recess at Philadelphia schools, and **why** (heat, cold, bad air, flash-flood rain, or dangerous wind)?

*Prototype, not official guidance. Always follow official National Weather Service alerts and your school district.*

## Step 1: Download Philadelphia's daily weather and air quality

We ask Open-Meteo (free, no API key) for one row per day in Philadelphia:
- **Weather** (archive API): hottest and coldest "feels like" temperature, total rain, and strongest wind gust.
- **Air quality** (air-quality API): PM2.5 (tiny particles) every hour. We average the hours into one number per day.

Note: Open-Meteo's air quality numbers are **estimates from a computer model (CAMS)**, not readings from a sensor. We say this in our write-up.

In [ ]:
import pandas as pd
import requests

LAT, LON = 39.95, -75.16            # Philadelphia (Center City)
START, END = "2022-08-01", "2026-11-12"

# Weather: one row per day
weather_url = "https://archive-api.open-meteo.com/v1/archive"
weather = requests.get(weather_url, params={
    "latitude": LAT, "longitude": LON,
    "start_date": START, "end_date": END,
    "daily": "apparent_temperature_max,apparent_temperature_min,precipitation_sum,wind_gusts_10m_max",
    "timezone": "America/New_York",
}).json()
weather_df = pd.DataFrame(weather["daily"])
weather_df["time"] = pd.to_datetime(weather_df["time"])

# Air quality: one row per hour, then averaged per day
air_url = "https://air-quality-api.open-meteo.com/v1/air-quality"
air = requests.get(air_url, params={
    "latitude": LAT, "longitude": LON,
    "start_date": START, "end_date": END,
    "hourly": "pm2_5",
    "timezone": "America/New_York",
}).json()
air_df = pd.DataFrame(air["hourly"])
air_df["time"] = pd.to_datetime(air_df["time"])
air_daily = air_df.resample("D", on="time")["pm2_5"].mean().reset_index()

# Put weather and air side by side, one row per day
df = weather_df.merge(air_daily, on="time")
print(df.shape)
df.head()

## Step 2: How many unsafe *school* days are there, and why?

A day is **unsafe** if any one of these happens. ⚠️ **Team to-do:** look up and cite a real source for each threshold (e.g. NWS, US EPA, or our school district's recess policy), and change the number if the source says something different.

| Warning | Rule |
|---|---|
| Too hot | "feels like" max ≥ 32 °C (≈90 °F, NWS "extreme caution") |
| Too cold | "feels like" min ≤ −7 °C (≈20 °F) |
| Bad air | daily PM2.5 > 35.4 µg/m³ (US EPA "Unhealthy for Sensitive Groups") |
| Flash flood / heavy rain | ≥ 25 mm (≈1 inch) of rain in the day |
| Dangerous wind (hurricanes, storms) | gusts ≥ 74 km/h (≈46 mph, NWS wind advisory level) |

**Why no tornado warning?** Our data doesn't record tornadoes, and they're very rare in Philadelphia, so a model has nothing to learn from. The "dangerous wind" warning covers the storms that bring them.

We only count **school days**: weekdays from September to June (no recess in July and August). We ignore holidays for now.

**Why this check comes first:** if a warning almost never happens, the model can't learn it. We'd drop it or change the threshold now, not on Sunday.

In [ ]:
df["too_hot"] = df["apparent_temperature_max"] >= 32
df["too_cold"] = df["apparent_temperature_min"] <= -7
df["bad_air"] = df["pm2_5"] > 35.4
df["heavy_rain"] = df["precipitation_sum"] >= 25
df["high_wind"] = df["wind_gusts_10m_max"] >= 74

hazards = ["too_hot", "too_cold", "bad_air", "heavy_rain", "high_wind"]
df["unsafe"] = df[hazards].any(axis=1)   # True if ANY hazard happens

is_weekday = df["time"].dt.weekday < 5           # Monday=0 ... Friday=4
is_school_month = ~df["time"].dt.month.isin([7, 8])
school = df[is_weekday & is_school_month].copy()
school["year"] = school["time"].dt.year

print("School days:", len(school))
print("Unsafe school days:", school["unsafe"].sum())
print()
print("Unsafe school days per year, by reason:")
print(school.groupby("year")[hazards + ["unsafe"]].sum())